# BITS Pilani — M.Tech AIML
# Machine Learning Assignment 2

**Name:** Adarsh Tayade  
**BITS ID:** 2026AJ05765

## 1. Problem Statement

The objective is to implement and compare five machine learning
classification models using a common dataset. Their performance
will be evaluated using six classification metrics.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer

print("Libraries imported successfully")

Libraries imported successfully


In [3]:
data = load_breast_cancer(as_frame=True)

df = data.frame.copy()

print("Dataset shape:", df.shape)

print("\nFirst five rows:")
display(df.head())

print("\nClass distribution:")
print(df["target"].value_counts())

print("\nMissing values:", df.isnull().sum().sum())

Dataset shape: (569, 31)

First five rows:


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0



Class distribution:
target
1    357
0    212
Name: count, dtype: int64

Missing values: 0


## 2. Dataset Description

The Breast Cancer Wisconsin Diagnostic dataset is a binary
classification dataset containing 569 instances, 30 numerical
input features, and one target column.

The target identifies two classes: malignant and benign.
The dataset meets the minimum requirement of 12 features
and 500 instances specified in the assignment.

## 3. Data Preprocessing and Train-Test Split

The dataset is split into training and testing sets using an
80:20 ratio. Stratified sampling preserves the class distribution
in both sets.

In [6]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (455, 30)
Testing features: (114, 30)
Training target: (455,)
Testing target: (114,)


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training data scaled:", X_train_scaled.shape)
print("Testing data scaled:", X_test_scaled.shape)

Training data scaled: (455, 30)
Testing data scaled: (114, 30)


## 4. Machine Learning Models

We implement and evaluate five classification algorithms:

1. Logistic Regression
2. Decision Tree Classifier
3. K-Nearest Neighbors (KNN)
4. Gaussian Naive Bayes
5. Support Vector Machine (SVM)

All models use the same training and testing split.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": CalibratedClassifierCV(
    SVC(random_state=42),
    ensemble=False
)
}

print("Five classification models defined successfully.")

Five classification models defined successfully.


## 5. Model Evaluation

The models are evaluated using Accuracy, AUC, Precision, Recall,
F1 Score, and Matthews Correlation Coefficient (MCC).

AUC is calculated using predicted class probabilities.
The test dataset is kept separate from model training.

In [11]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

results = []
trained_models = {}

for name, model in models.items():

    # Use scaled features for scale-sensitive models
    if name in ["Logistic Regression", "KNN", "Support Vector Machine"]:
        model.fit(X_train_scaled, y_train)
        predictions = model.predict(X_test_scaled)
        probabilities = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        probabilities = model.predict_proba(X_test)[:, 1]

    trained_models[name] = model

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "AUC": roc_auc_score(y_test, probabilities),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1 Score": f1_score(y_test, predictions, zero_division=0),
        "MCC": matthews_corrcoef(y_test, predictions)
    })

results_df = pd.DataFrame(results)

display(results_df.round(4))

,Model,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,Logistic Regression,0.9825,0.9954,0.9861,0.9861,0.9861,0.9623
1,Decision Tree,0.9123,0.9157,0.9559,0.9028,0.9286,0.8174
2,KNN,0.9561,0.9788,0.9589,0.9722,0.9655,0.9054
3,Naive Bayes,0.9386,0.9878,0.9452,0.9583,0.9517,0.8676
4,Support Vector Machine,0.9825,0.9950,0.9861,0.9861,0.9861,0.9623


In [12]:
print("Model performance observations:\n")

for _, row in results_df.iterrows():
    print(
        f"{row['Model']}: "
        f"Accuracy={row['Accuracy']:.4f}, "
        f"AUC={row['AUC']:.4f}, "
        f"F1={row['F1 Score']:.4f}, "
        f"MCC={row['MCC']:.4f}"
    )

winner = results_df.loc[results_df["F1 Score"].idxmax(), "Model"]
print("\nModel with the highest F1 Score:", winner)

Model performance observations:

Logistic Regression: Accuracy=0.9825, AUC=0.9954, F1=0.9861, MCC=0.9623
Decision Tree: Accuracy=0.9123, AUC=0.9157, F1=0.9286, MCC=0.8174
KNN: Accuracy=0.9561, AUC=0.9788, F1=0.9655, MCC=0.9054
Naive Bayes: Accuracy=0.9386, AUC=0.9878, F1=0.9517, MCC=0.8676
Support Vector Machine: Accuracy=0.9825, AUC=0.9950, F1=0.9861, MCC=0.9623

Model with the highest F1 Score: Logistic Regression


In [13]:
import os
import joblib

os.makedirs("models", exist_ok=True)

for name, model in trained_models.items():
    filename = name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(model, os.path.join("models", filename))

joblib.dump(scaler, "models/scaler.joblib")

print("All models saved successfully!")

All models saved successfully!


In [14]:
test_data = X_test.copy()
test_data["target"] = y_test

test_data.to_csv("test_data.csv", index=False)

print("Test CSV created successfully!")
print("Dataset shape:", test_data.shape)

Test CSV created successfully!
Dataset shape: (114, 31)


In [15]:
print("Model files:", os.listdir("models"))
print("Test CSV exists:", os.path.exists("test_data.csv"))

Model files: ['logistic_regression.joblib', 'decision_tree.joblib', 'knn.joblib', 'naive_bayes.joblib', 'support_vector_machine.joblib', 'scaler.joblib']
Test CSV exists: True
